# 04 — Compare binary and hybrid NOUL across both JEV models

**Purpose:** evaluate all four saved model/question-design arms side by side, using the same citation cohort and human screening reference. `True` means retain for full-text assessment, including uncertain eligibility; a false negative is a human-retain citation excluded at a stated cutoff.

This notebook is read-only by default: **no API calls, API key, model execution or automatic recovery**. The two archive sources are explicit in `configs/evaluations/comparison.yaml`, independent of the central default approach. Existing paid responses and raw probabilities are preserved. No invented data or missing arm is represented as observed model performance.

Archived binary wording differs from hybrid v1. Report that limitation with any apparent differences; this comparison cannot isolate question design alone. No threshold is adopted by running exploratory calculations.

## 1. Load both approaches and both models

Run from a fresh project `.venv` kernel. The loader verifies input, reference, cohort, model and saved-probability integrity. Missing results remain visible. **Inputs:** archived full runs and prepared data. **Output:** the in-memory `comparison` and four unique arm labels used consistently in tables and figures.

In [ ]:
from jev_eval.comparison import load_screening_comparison
from jev_eval.environment import find_project_root

COMPARISON_CONFIG = "configs/evaluations/comparison.yaml"
root = find_project_root()
comparison = load_screening_comparison(root, COMPARISON_CONFIG)

display(comparison.tables["arms"])
print("Target recall:", comparison.settings.target_recall)

## 2. Confirm the comparison design and human reference

All four arms must share ordered study IDs, citation inputs and reference CSV. Comparison is blocked if they differ. Human inclusion status and ITEM_ID are not model input fields. Original study ID is the join key.

Within each approach, both models use the same saved question definitions. Across approaches, the exact overall definitions are compared rather than assumed equivalent. Existing binary outputs are not relabelled as reviewed-v1 outputs. Human screening is the comparison reference; workbook discrepancies need review rather than automatic adjudication.

In [ ]:
display(comparison.tables["comparability"])
reference = comparison.datasets["binary"].references
print("Reference SHA-256:", reference.sha256)
display(reference.summary())
print("Human retain:", sum(comparison.datasets["binary"].reference_labels.values()))
print(
    "Human exclude:",
    sum(not value for value in comparison.datasets["binary"].reference_labels.values()),
)

## 3. Inspect all four denominators and missing predictions

Do this before interpreting performance. Successful, failed, unknown, textless and missing records remain distinct. An interrupted study deliberately skipped after review remains unscored, not human-exclude. Scored-pair metrics omit unscored studies; cohort recall gives them no model credit. Every unscored study stays in a separate manual-review queue.

A recorded completion status does not prove all expected rows were saved. No missing prediction is filled from another approach or model, and this notebook makes no requests to repair it.

In [ ]:
coverage_columns = [
    "arm",
    "expected_records",
    "recorded_rows",
    "successful",
    "failed",
    "unknown",
    "not_inferred_no_text",
    "missing",
    "invalid_probabilities",
    "pending_review",
    "status",
]
display(comparison.tables["coverage"][coverage_columns])

## 4. Understand the available scores and archived questions

The primary four-arm comparison uses each binary overall answer and each hybrid `overall_eligibility` answer. Hybrid criterion scores do not become extra models. Score summaries are not criterion-level accuracy: no separate human labels for each criterion are assumed.

No calibrated joint probability is inferred from independent NOUL answers. Review the exact archived instructions if explaining why a citation was retained or excluded.

In [ ]:
display(comparison.tables["question_summary"])
display(comparison.tables["criteria"])

## 5. Compare ROC AUC, PR AUC and average precision

These ranking metrics need no cutoff. They use valid scored human-labelled pairs and require both classes. PR AUC here is trapezoidal; average precision (AP) is non-interpolated. Neither guarantees high recall at a particular cutoff.

The first table uses each arm's available scores. The second restricts scoring to the **identical set of studies scored by all four arms**, so unequal missingness cannot change its comparison set. Inspect both counts. Results remain exploratory and may reflect different eligibility wording.

In [ ]:
auc_columns = [
    "model",
    "roc_auc",
    "pr_auc_trapezoidal",
    "average_precision",
    "scored_labelled_pairs",
    "positive_pairs",
    "negative_pairs",
    "unscored_studies",
    "status",
]
display(comparison.tables["ranking_metrics"][auc_columns].round(4))
print("Same all-four-scored subset:")
display(
    comparison.tables["common_scored_ranking"]
    .reindex(columns=auc_columns + ["common_scored_studies"])
    .round(4)
)

## 6. Explore recall, precision, specificity, F2 and WSS@95

For each arm separately, choose the highest observed cutoff achieving the configured target (initially 95%) among **all human-retain studies**, giving unscored studies no model credit. Inclusion uses score >= cutoff, including ties. If unattainable, report that rather than dropping missing positives. The 0.5 baseline is a reference, not a validated choice.

Precision/PPV, specificity and F2 use scored labelled pairs. Cohort WSS@95 is automatic-exclusion fraction minus 0.05 when the full-cohort recall target is met; unscored studies remain in the review queue. This is not measured staff time and differs from the protocol's rank-based expression pending review.

Cutoffs are selected and assessed on these same labels: **exploratory, not independent validation**. They can differ across the four arms. No official threshold or inference configuration is changed, and rounded values must not be substituted for exact cutoffs.

In [ ]:
metric_columns = [
    "model",
    "cutoff_source",
    "threshold",
    "recall_scored",
    "recall_cohort_lower_bound",
    "specificity_scored",
    "ppv_scored",
    "f2_scored",
    "wss95_scored",
    "wss95_cohort_conservative",
    "review_queue_with_unscored",
    "unscored_studies",
]
performance = comparison.tables["screening_metrics"].reindex(columns=metric_columns)
display(
    performance.style.format(
        {
            "threshold": "{:.17g}",
            "recall_scored": "{:.2%}",
            "recall_cohort_lower_bound": "{:.2%}",
            "specificity_scored": "{:.2%}",
            "ppv_scored": "{:.2%}",
            "f2_scored": "{:.3f}",
            "wss95_scored": "{:.2%}",
            "wss95_cohort_conservative": "{:.2%}",
        },
        na_rep="not available",
    )
)

## 7. View all four arms together

ROC/PR curves use scored pairs. Solid recall curves use the full-labelled lower bound; dotted curves use scored-pair recall. The workload curve includes erroneous exclusions as well as correct exclusions. Candidate matrices show scored-pair counts only; all unscored records remain outside them and in the manual queue.

Each figure labels both model and NOUL approach. The four matrices use their own exploratory target cutoffs; they do not imply a shared adopted cutoff.

In [ ]:
from jev_eval.evaluation import threshold_tradeoff_curve
from jev_eval.reporting import (
    plot_probability_curves,
    plot_recall_candidate_matrices,
    plot_threshold_tradeoffs,
)

figures = {}
roc_pr = plot_probability_curves(comparison.ranking)
if roc_pr is not None:
    figures["four-arm-roc-pr"] = roc_pr
    display(roc_pr)
else:
    print("ROC/PR curves need both human reference classes among scored pairs.")
figures["four-arm-recall-workload"] = plot_threshold_tradeoffs(
    threshold_tradeoff_curve(comparison.data),
    comparison.candidates,
    comparison.settings.target_recall,
)
display(figures["four-arm-recall-workload"])
matrices = plot_recall_candidate_matrices(comparison.candidates)
if matrices is not None:
    figures["four-arm-confusion-matrices"] = matrices
    display(matrices)
else:
    print("No attainable target cutoffs; confusion matrices withheld.")

## 8. Secondary hybrid R2/R3 analyses and protocol status

R2 is the minimum and R3 the mean of the first four primary-domain scores. Controls and outcomes are secondary. These are derived scores, not calibrated joint probabilities and not additional primary experimental arms. R1 remains disabled pending protocol clarification.

Formal tuning requires the original study-ID-linked development/tuning membership and reconciliation of the 7,008/7,000 cohort discrepancy. These pooled exploratory candidates do not replace that split or select a final winning rule. The planned 2,000 unseen studies are not loaded or used here.

In [ ]:
from jev_eval.screening import load_screening_settings

print("R1:", load_screening_settings(root).r1_policy)
display(comparison.tables["hybrid_derived_scores"])

## 9. Compare missed human-retain studies on paired citations

For each of the six arm pairs, count studies missed by both, by one, or by neither, using their exact target cutoffs. Only human-retain studies scored in both arms enter these counts; unavailable positives are reported separately. This is an aggregate audit, not a significance test or independently validated comparison.

In [ ]:
display(comparison.tables["paired_positive_errors"])

## 10. Optional comparison export — writes files

**File-writing warning:** enabling both `allow_writes` in the comparison YAML and `EXPORT_COMPARISON` below creates a new private aggregate HTML/CSV/figure bundle under its configured results folder, with input/configuration/source and original-run provenance. Existing inference files are never overwritten. No API calls are made. Review exported material before sharing.

In [ ]:
from jev_eval.comparison import export_screening_comparison

EXPORT_COMPARISON = False
if EXPORT_COMPARISON:
    comparison_directory = export_screening_comparison(comparison, figures, allow_writes=True)
    print("Private comparison export:", comparison_directory)
else:
    print("Export disabled; all calculations remain in memory.")